# 🔥 01i — Pipeline incendies de forêt (BDIFF)

Construit `dim_incendies.parquet` (dept × année-mois). Lancer `00_config_commun.ipynb`
avant.

Source : BDIFF (Base de Données sur les Incendies de Forêts en France),
Ministère de l'Agriculture / IGN.
https://bdiff.agriculture.gouv.fr/incendies

Pas de fichier bulk téléchargeable directement (le site est une appli JS,
pas un jeu de données data.gouv.fr classique). Le CSV a été obtenu à la main
via le formulaire de recherche du site : sur https://bdiff.agriculture.gouv.fr/incendies,
critère Dates du 01/01/2020 au 31/12/2025 + "France entière" cochée, bouton
"Filtrer" puis bouton "CSV" sous le tableau de résultats (télécharge un zip
contenant le CSV + 2 PDF de mentions légales/définitions, à ignorer).
Fichier : `data/raw/incendies/Incendies_2020_2025.csv` (16 356 incendies,
2020-2025, 96 départements métropolitains + DOM).

Le fichier a 3 lignes d'en-tête avant la vraie ligne de colonnes (nombre
total d'incendies + critères de sélection appliqués) — variable d'un export
à l'autre selon qu'un bandeau d'avertissement s'affiche ou non, d'où la
détection de la ligne d'en-tête par son contenu (`"Année;"`) plutôt qu'un
`skiprows` fixe.

Grain le plus fin possible : la date de première alerte, pas de notion de
mois dans les données brutes — on regroupe nous-mêmes par dept × année-mois.
Pour l'instant, la table est laissée telle quelle en sortie du groupby (pas
de grille dept × année-mois complétée avec des 0) : un dept-mois sans ligne
veut juste dire "pas d'incendie recensé", sans trancher si c'est un vrai
zéro ou une valeur à traiter comme manquante — ce choix est fait plus tard,
au moment du nettoyage (`04_nettoyage_donnees.ipynb`).

Colonnes produites :
- `nb_incendies` : nombre d'incendies recensés dans le mois
- `surface_parcourue_ha` : surface totale parcourue par le feu dans le mois (hectares)

In [ ]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

In [2]:
def build_dim_incendies() -> pd.DataFrame:
    """
    Construit dim_incendies à partir de l'export BDIFF.

    CLÉ PRIMAIRE : dept × annee_mois
    COLONNES : nb_incendies, surface_parcourue_ha
    """
    fpath = RAW_DIR / "incendies" / "Incendies_2020_2025.csv"
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    with open(fpath, encoding="utf-8") as f:
        lignes = f.readlines()
    idx_entete = next(i for i, l in enumerate(lignes) if l.startswith("Année;"))

    df = pd.read_csv(fpath, sep=";", encoding="utf-8", skiprows=idx_entete,
                      engine="python", quotechar='"')
    print(f"  Brut : {len(df):,} incendies")

    df["dept"] = df["Département"].astype(str).str.strip().str.upper().str.zfill(2)
    df = df[df["dept"].isin(DEPTS)]
    print(f"  Après filtre DEPTS (hors DOM) : {len(df):,} incendies")

    df["date_alerte"] = pd.to_datetime(df["Date de première alerte"])
    df["annee_mois"] = df["date_alerte"].dt.strftime("%Y-%m")
    df = df[(df["date_alerte"].dt.year >= ANNEE_DEBUT) & (df["date_alerte"].dt.year <= ANNEE_FIN)]

    agg = df.groupby(["dept", "annee_mois"]).agg(
        nb_incendies=("Numéro", "count"),
        surface_parcourue_ha=("Surface parcourue (m2)", lambda s: s.sum() / 10_000),
    ).reset_index()

    # Pas de complétion de grille dept x année-mois ici : on laisse tel quel
    # (dept-mois sans ligne = pas d'incendie recensé, à trancher zéro vs NaN
    # au moment du nettoyage).
    print(f"  ✅ dim_incendies : {agg.shape[0]:,} lignes × {agg.shape[1]} colonnes")
    print(f"  Départements : {agg['dept'].nunique()} | Mois : {agg['annee_mois'].min()} → {agg['annee_mois'].max()}")
    return agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_incendies = build_dim_incendies()

if not dim_incendies.empty:
    valider_dim_table(dim_incendies, "dim_incendies", cle=["dept", "annee_mois"])
    dim_incendies.to_parquet(TABLES_DIR / "dim_incendies.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_incendies.parquet")
    display(dim_incendies.head(10))

  Brut : 16,356 incendies
  Après filtre DEPTS (hors DOM) : 16,007 incendies
  ✅ dim_incendies : 2,421 lignes × 4 colonnes
  Départements : 92 | Mois : 2020-01 → 2025-12
── Validation de dim_incendies ──
  ✅ Tous les codes dept sont valides (92 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee_mois']
  ✅ OK — prêt pour la fusion (dim_incendies)


✅ Sauvegardé → data/processed/dim_incendies.parquet


,dept,annee_mois,nb_incendies,surface_parcourue_ha
0,01,2020-08,2,13.000
1,01,2021-03,1,1.000
2,01,2022-04,1,3.000
3,01,2022-06,1,1.500
4,01,2022-08,5,10.000
5,01,2024-08,1,2.000
6,01,2024-09,1,1.025
7,01,2025-08,2,10.000
8,02,2020-06,2,5.000
9,02,2020-07,2,20.500
